# Tips: exploratory restaurant bill analysis
Pamela Vilchez | Applied graduate coursework

How do bills and tips vary in a teaching sample? This focused portfolio adaptation retains the distribution, bill/tip relationship and day-of-week questions from the recovered visualization submission. It repairs the missing data-loading step and replaces deprecated distplot calls with histplot. Demographic subgroup significance claims are omitted because the assignment did not conduct formal tests. No staffing or revenue impact is claimed.

Supply an authorized `data/tips.csv`. Raw records are excluded pending source and redistribution review.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

tips = pd.read_csv('data/tips.csv')
assert {'total_bill','tip','day'}.issubset(tips.columns)
assert not tips[['total_bill','tip','day']].isna().any().any()
figures=Path('figures')
figures.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')
summary={'rows':len(tips),'bill_tip_correlation':tips['total_bill'].corr(tips['tip']),
         'median_bill':tips['total_bill'].median(),'median_tip':tips['tip'].median(),
         'mean_bill':tips['total_bill'].mean(),'mean_tip':tips['tip'].mean()}
print(pd.Series(summary).to_string())
daily=tips.groupby('day').agg(observations=('tip','size'), mean_bill=('total_bill','mean'),mean_tip=('tip','mean'))
print(daily.round(3).to_string())
summary['daily']=daily.reset_index().to_dict(orient='records')
Path('results.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')

rows                    244.000000
bill_tip_correlation      0.675734
median_bill              17.795000
median_tip                2.900000
mean_bill                19.785943
mean_tip                  2.998279
      observations  mean_bill  mean_tip
day                                    
Fri             19     17.152     2.735
Sat             87     20.441     2.993
Sun             76     21.410     3.255
Thur            62     17.683     2.771


In [2]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
sns.histplot(data=tips, x='total_bill', bins=20, ax=axes[0], color='#276e90')
sns.histplot(data=tips, x='tip', bins=20, ax=axes[1], color='#276e90')
axes[0].set(xlabel='Total bill (source dollar units)', title='Bill distribution')
axes[1].set(xlabel='Tip (source dollar units)', title='Tip distribution')
fig.tight_layout()
fig.savefig(figures/'bill-tip-distributions.svg')
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
order=['Thur','Fri','Sat','Sun']
daily.reindex(order)['mean_bill'].plot.bar(ax=axes[0],color='#276e90',rot=0)
daily.reindex(order)['observations'].plot.bar(ax=axes[1],color='#276e90',rot=0)
axes[0].set(ylabel='Mean bill (source dollar units)',xlabel='Recorded day',title='Sample mean bill by day')
axes[1].set(ylabel='Observations',xlabel='Recorded day',title='Unequal sample sizes by day')
fig.tight_layout()
fig.savefig(figures/'day-comparison.svg')
plt.close(fig)

## Interpretation and limitations
The bill/tip correlation describes association; it does not estimate a causal effect. Day averages should be read together with sample counts, since observation counts differ. These records cannot establish total restaurant revenue or customer traffic for each day without a complete sampling frame. The charts report descriptive distributions, not significance tests. Dataset provider, collection design and redistribution terms must be confirmed before public release.

For a real business question, collect representative observations across shifts and account for party size, service and repeated visits. The portfolio copy intentionally focuses on descriptive evidence supported by this assignment.